# Project 6: Ethics & Bias Audit

**Dataset:** California Housing

**Goal:** Examine whether our ML model treats different geographic regions fairly. This project connects to A4.4 (Ethical Implications) and TOK.

In [ ]:
# ====== Setup: imports + data loader ======
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120

DATA_URL = 'https://raw.githubusercontent.com/lucyyuhongxia-gmail/A4_ML_Projects/main/datasets'
def load_data(fn):
    """Load a real dataset from GitHub."""
    df = pd.read_csv(f"{DATA_URL}/{fn}")
    print(f"Loaded: {fn} -> {df.shape[0]:,} rows x {df.shape[1]} cols")
    return df
print('Setup OK!')
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
import time

## 1. Load Data & Define Sensitive Attributes

We simulate protected attributes by splitting the data along geographic lines (North/South California). In real-world applications, these could be race, gender, zip code, or socioeconomic status.

In [ ]:
# Create simulated sensitive attributes — 模拟敏感属性
# 在真实应用中可能是: 种族、性别、邮编、收入阶层
df = load_data('housing.csv')

# Use Latitude as proxy for region
lat_median = df['Latitude'].median()
df['Region'] = np.where(df['Latitude'] >= lat_median, 'North', 'South')
df['IncomeGroup'] = pd.qcut(df['MedInc'], q=3, labels=['Low','Middle','High'])

print('Region distribution:')
print(df['Region'].value_counts().to_string())
print('\nIncome distribution:')
print(df['IncomeGroup'].value_counts().to_string())

## 2. Train Model & Measure Regional Bias

In [ ]:
# Train a linear regression model — 训练线性回归模型
X = df[['MedInc','HouseAge','AveRooms','AveBedrms',
        'Population','AveOccup','Latitude','Longitude']]
y = df['MedHouseVal']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f'Overall R²: {r2_score(y_test, y_pred):.4f}')

# Analyze errors by region — 按区域分析预测误差
test_df = X_test.copy()
test_df['Actual'] = y_test
test_df['Predicted'] = y_pred
test_df['Region'] = df.loc[test_df.index, 'Region'].values
test_df['Error'] = test_df['Predicted'] - test_df['Actual']

print('\nPer-Region Error Analysis:')
for region in ['North', 'South']:
    sub = test_df[test_df['Region'] == region]
    rmse = np.sqrt(mean_squared_error(sub['Actual'], sub['Predicted']))
    print(f'  {region}: RMSE={rmse:.4f}, Mean Error={sub["Error"].mean():+.4f}')

north_err = test_df[test_df['Region']=='North']['Error'].mean()
south_err = test_df[test_df['Region']=='South']['Error'].mean()
print(f'\n⚠️  Systematic bias detected!')
print(f'   North overpredicted by ${north_err*100:.0f}K on average')
print(f'   South underpredicted by ${south_err*100:.0f}K on average')

## 3. Visualizing Bias

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot by region
sns.boxplot(data=test_df, x='Region', y='Error', ax=axes[0])
axes[0].axhline(y=0, color='r', ls='--')
axes[0].set_title('Prediction Error by Region')
axes[0].set_ylabel('Error = Predicted - Actual ($100K)')

# Boxplot by income group
test_df['IncomeGroup'] = df.loc[test_df.index, 'IncomeGroup'].values
sns.boxplot(data=test_df, x='IncomeGroup', y='Error', ax=axes[1],
            order=['Low','Middle','High'])
axes[1].axhline(y=0, color='r', ls='--')
axes[1].set_title('Prediction Error by Income Group')
plt.tight_layout(); plt.show()

## 4. Environmental Impact

In [ ]:
# Every ML training run has a carbon cost
# 每一次训练都有碳成本 — 我们有多频繁地在做不必要的实验？
start = time.time()
scaler = StandardScaler()
GridSearchCV(Ridge(), {'alpha': [0.01, 0.1, 1, 10]}, cv=3).fit(
    scaler.fit_transform(X), y)
elapsed = time.time() - start

print(f'This Grid Search: {elapsed:.1f}s on CPU')
print(f'  Est. energy: {elapsed*65/3600:.6f} kWh')
print(f'  Est. CO2:    {elapsed*65/3600*0.5:.6f} kg')
print()
print('For neural network training (GPU, 24h):')
print('  Energy: ~7.2 kWh')
print('  Est. CO2: ~3.6 kg — equivalent to driving ~15 km!')

## 5. Discussion Questions (Group or Written)

1. **Bias:** Our model overprices the North and underprices the South by ~$2K on average. If this model were used for property tax assessment, would residents in the North pay more than they should? Is this fair?

2. **Privacy:** The dataset contains precise latitude/longitude coordinates. What could a malicious actor infer about specific individuals? Would aggregate data (e.g., zip code level) be sufficient?

3. **Accountability:** If a biased model causes harm, who is responsible — the data scientist who built it, the school that deployed it, or the algorithm itself?

4. **Environmental:** Each Grid Search we run consumes electricity. How many Grid Searches are worth the carbon cost? Should ML training be taxed based on energy consumption?

5. **Societal:** Automated property valuation systems are already used by Zillow and Redfin. Do these systems help or harm low-income neighborhoods? What safeguards would you propose?